In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import torch
import warnings
import os

# Отключаем все предупреждения
warnings.filterwarnings('ignore')

# Отключаем конкретное предупреждение HuggingFace про симлинки в Windows
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
# Загружаем модель
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Используемое устройство: {device}")

model = SentenceTransformer('intfloat/multilingual-e5-small', device=device)

C:\Users\Gulshat\AppData\Roaming\Python\Python39\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Используемое устройство: cpu


In [2]:
train = pd.read_parquet(r"C:\Users\Gulshat\Downloads\NLP_avito_interns\NLP_avito_interns\dataset\train.parquet")
queries = pd.read_parquet(r"C:\Users\Gulshat\Downloads\NLP_avito_interns\NLP_avito_interns\dataset\benchmark_queries.parquet")
items = pd.read_parquet(r"C:\Users\Gulshat\Downloads\NLP_avito_interns\NLP_avito_interns\dataset\benchmark_items.parquet")

print(f"Queries: {queries.shape}")
print(f"Items: {items.shape}")

Queries: (2452, 6)
Items: (189212, 14)


In [3]:
# Заполняем пропуски
queries['search_query'] = queries['search_query'].fillna('').astype(str)
queries['search_infm_params_text'] = queries['search_infm_params_text'].fillna('').astype(str)
items['item_title_raw'] = items['item_title_raw'].fillna('').astype(str)
items['item_description_raw'] = items['item_description_raw'].fillna('').astype(str)
items['item_infm_params_text'] = items['item_infm_params_text'].fillna('').astype(str)

# Склеиваем и обрезаем до 128 символов
queries['text'] = ("query: " + queries['search_query'] + " " + queries['search_infm_params_text']).str.slice(0, 128)
items['text'] = ("passage: " + items['item_title_raw'] + " " + items['item_description_raw'] + " " + items['item_infm_params_text']).str.slice(0, 128)

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Используемое устройство: {device}")

model = SentenceTransformer('intfloat/multilingual-e5-small', device=device)
# Жестко ограничиваем длину для модели
model.max_seq_length = 128 

Используемое устройство: cpu


In [5]:
print("Кодируем объявления (ускоренный режим)...")

item_embeddings = model.encode(
    items['text'].tolist(), 
    batch_size=256, # Увеличили батч для скорости
    show_progress_bar=True, 
    normalize_embeddings=True
)

# Сохраняем на диск (страховка от падения)
np.save('item_embeddings.npy', item_embeddings)
print("Готово! Размер эмбеддингов:", item_embeddings.shape)

Кодируем объявления (ускоренный режим)...


Batches: 100%|██████████| 740/740 [50:56<00:00,  4.13s/it]


Готово! Размер эмбеддингов: (189212, 384)


In [6]:
print("Кодируем запросы...")

query_embeddings = model.encode(
    queries['text'].tolist(), 
    batch_size=256, 
    show_progress_bar=True, 
    normalize_embeddings=True
)
print("Готово! Размер эмбеддингов запросов:", query_embeddings.shape)

Кодируем запросы...


Batches: 100%|██████████| 10/10 [00:13<00:00,  1.36s/it]

Готово! Размер эмбеддингов запросов: (2452, 384)


In [7]:
print("Считаем сходство и ищем топ-50...")

top_n = 50
batch_size = 200
item_ids = items['item_id'].values
all_predictions = []

for start_idx in range(0, len(query_embeddings), batch_size):
    end_idx = min(start_idx + batch_size, len(query_embeddings))
    batch_queries = query_embeddings[start_idx:end_idx]
    
    # Косинусное сходство через скалярное произведение (так как вектора нормализованы)
    sims = batch_queries @ item_embeddings.T
    
    # Сортируем и берем топ-50
    top_indices = np.argsort(sims, axis=1)[:, -top_n:][:, ::-1]
    
    for row in top_indices:
        all_predictions.append(item_ids[row].tolist())
        
    print(f"Обработано: {end_idx} / {len(query_embeddings)}")

Считаем сходство и ищем топ-50...
Обработано: 200 / 2452
Обработано: 400 / 2452
Обработано: 600 / 2452
Обработано: 800 / 2452
Обработано: 1000 / 2452
Обработано: 1200 / 2452
Обработано: 1400 / 2452
Обработано: 1600 / 2452
Обработано: 1800 / 2452
Обработано: 2000 / 2452
Обработано: 2200 / 2452
Обработано: 2400 / 2452
Обработано: 2452 / 2452


In [10]:
answer = pd.DataFrame({
    'query_id': queries['query_id'],
    'answer': [' '.join(map(str, top50)) for top50 in all_predictions]
})

answer.to_csv('answer.csv', index=False, encoding='utf-8')
